<h3>Student Name:</h3>
<p>Sanyam Goel</p>

<h1>Assignment 3</h1>
<h2>Part 1</h2>
<p>

<ol>
    <li>
        <strong>Vocabulary:</strong> The vocabulary of VSM is a composed of TermStatistics Dictionary
    </li>
    <li>
        <strong>TermStatistics/ Dictionary Value:</strong> Contains two attributes, <strong>frequency</strong> and
        <strong>idf</strong> value of that term
    </li>
    <li>
        <strong> Dictionary Key:</strong> The term itself is the key of the dictionary
    </li>
</ol>
</p>

<h2>TermStatistics class</h2>
<p>
    This class encapsulates two properties
<ol>
    <li>
        <strong>frequency: </strong>Frequency of the term in the vocabulary
    </li>
    <li>
        <strong>idf: </strong> The IDF value of the correcspnding term in dictionary
    </li>
</ol>
The properties use dunder variables and are exposed by public getter methods to highlight direct access should not be
made
</p>
<h2> VectorSpaceModel class</h2>
<p>
    In order to improve the readability and maintainability of the code, this class uses a lot of dunder methods meant
    for particular purposes and encapsulates all variables using dunder vars. The public apis are available for specific
    attributes apart from two methods required in the assignment.
    <br />
    <strong>These dunder methods or granular method helps in reusabality for majority of the code during extendion( like in case of VSMExtended all the methods are called from super(reused) as it is, with the addition of a new method to filter stop words only</strong>

<h4>Following are the attriutes</h4>
<ul>
    <li>
        <strong>vocabulary: </strong>It is a dictionary of type dict[str, TermStatistics]
    </li>
    <li>
        <strong>enable_lowercase: </strong> To enable lowercase when creating vocabulary and calculating tf-idf on
        documents
    </li>
    <li>
        <strong>v_max: </strong>Limits the maximum number of words in the vocabulary, ordered by frequency
    </li>
    <li>
        <strong>enable_stripping: </strong> If enabled removes string.punctuation from all dictionary vocabulary at the
        time of creation of VSM and also while processing documents o calculate tf-idf
    </li>
</ul>

<h4>Following methods are exposed</h4>
<ul>
    <li>
        <strong>fit(): </strong> Creates a <strong>sorted</strong> vocabulary considering stripping and lowercase values
    </li>
    <li>
        <strong>transform(): </strong> It transforms a set of documents into vectors via df-idf based on vocabulary idf
        values
    </li>
    <li>
        <strong>v_max: </strong>Limits the maximum number of words in the vocabulary, ordered by frequency
    </li>
    <li>
</ul>

<h4>Following dunder methods are also available </h4>
<ul>
    <li>
        <strong>__parse_words__():</strong> Parses sentences to create a list of list of words
    </li>
    <li>
        <strong>__create_vocabulary_frequency_dictionary__(parsed_words): </strong> Adds Frequency for vocabulary
    </li>
    <li>
        <strong>__sort_vocabulary__(): </strong> Sorts vocabulary based on Frequency
    </li>
    <li>
        <strong>__truncate_vocabulary__(): </strong> Truncates vocabulary based per v_max value
    </li>
    <li>
        <strong>__calculate_vocabulary_idf__(sentences)</strong>_ Calculates and updates IDF values for the vocabulary
    </li>
</ul>
</p>

<h1>Important Notes:</h1>
<p style='background-color: yellow'>Since the vocabulary sequence may vary from the examples due to sort order, the tf-idf documents results will also be differently ordered based on the word appearence in the vocabulary</p>

In [1]:
# This class is utilized by vector state machine when creating vocabulary to store frequecny and idf of each term.
class TermStatistics:
    
    def __init__(self, frequency:int, idf:float):
        
        self.__frequency__ = frequency
        self.__idf__ = idf

    def get_frequency(self)-> int:
        
        return self.__frequency__

    def get_idf(self)-> float:
        
        return self.__idf__

    def set_frequency(self,frequency:int)-> None:
        
        self.__frequency__ = frequency

    def set_idf(self,idf:float)-> None:
        
        self.__idf__ = idf

    def __repr__(self)->str:

        return  str.format('''Frequency:' {frequency} idf: {idf} \n''', frequency=str(self.__frequency__), idf=str(self.__idf__))

In [2]:
import math
import string

#Vector state model 
class VectorSpaceModel:

    def __init__(self, enable_lowercase:bool, v_max:int, enable_stripping:bool):
     
        if(v_max <= 0):
            raise Exception('Invalid VMax')
            
        self.__vocabulary__:dict[str, TermStatistics] = dict()
        self.__enable_lowercase__:bool = enable_lowercase;
        self.__v_max__:int=v_max
        self.__enable_stripping__:bool = enable_stripping

    def fit(self, sentences:list[str])-> None:
        
        parsed_words:list[str]= self.__parse_words__(sentences)
        self.__create_vocabulary_frequency_dictionary__(parsed_words)
        self.__sort_vocabulary__()
        self.__truncate_vocabulary__();
        self.__calculate_vocabulary_idf__(sentences)


    # This method parses words from documents and converts into a list
    def __parse_words__(self, documents:list[str]):

        return [self.__process_word__(word)
                for words in [document.split() for document in documents] 
                for word in words ]

    # This method strips punctuations from a word and convert to lowercase if lowercase is enabled
    def __process_word__(self, word:str)-> None:

        processed_word = word;
        
        if self.__enable_lowercase__:
            processed_word= processed_word.lower()

        if self.__enable_stripping__:
            processed_word= processed_word.strip(string.punctuation)
            
        return processed_word

    # This method creates a dictionary and updates frequency of each term
    def __create_vocabulary_frequency_dictionary__(self, parsed_words:list[str]):
        
        for word in parsed_words:
            if word in self.__vocabulary__.keys():
                word_frequency= self.__vocabulary__[word].get_frequency();
                self.__vocabulary__[word].set_frequency(word_frequency+1)
            else:
                self.__vocabulary__[word]= TermStatistics(1,0.0)

    # This method sorts a dictionary based on keys ( since Python 3.10.X a dictionary keys are ordered and thereofore sort order is preserved
    def __sort_vocabulary__(self)-> None:
        
        self.__vocabulary__= dict(sorted(self.__vocabulary__.items(), key= lambda word: word[1].get_frequency(), reverse=True))

    # Thsi methods truncates vocabulary based on v_max
    def __truncate_vocabulary__(self)-> None:
        
        if self.__v_max__ < len(self.__vocabulary__):
            removable_keys=[*self.__vocabulary__.keys()][self.__v_max__:]
            self.__vocabulary__ = { key:self.__vocabulary__[key] for key in self.__vocabulary__.keys() if key not in removable_keys }

    # Calculates the idf value of the dictionary
    def __calculate_vocabulary_idf__(self, sentences:list[str])-> None:

        total_documents:int = len(sentences)

        for key in self.__vocabulary__.keys():
            
            number_of_docs_containing_key=0;

            for sentence in sentences:
                
                if key in [self.__process_word__(word) for word in  sentence.split()]:
                    number_of_docs_containing_key = number_of_docs_containing_key+1
            
            # formula for idf       
            idf = math.log((1+total_documents)/(number_of_docs_containing_key + 1))+1
            self.__vocabulary__[key].set_idf(idf)

    # Transforms documents into df-idf vectors using the vocabulary
    def transform(self, sentences:list[str])-> list[list[float]]:
        
        words_list:list[list[str]]= [words for words in [sentence.split() for sentence in sentences]]
        
        vectors:list[list[float]]= list(list())
        
        for words in words_list:
            
            vector:list[float]= list()
            tf:float=0.0
            for term in self.__vocabulary__:

                processed_words= [self.__process_word__(elem) for elem in words]
                tf = processed_words.count(term)
                tf_idf = tf* self.__vocabulary__[term].get_idf()
                vector.append(tf_idf)
            vectors.append(vector)
            
        return vectors

    # Returns the created vocabulary
    def get_vocabulary(self)-> dict[str,TermStatistics]: 
        
        return self.__vocabulary__

    # Returns stripping status
    def is_stripping_enabled(self)-> bool:
        
        return self.__enable_stripping__

    # Returns lowercase enable status
    def is_lowercase_enabled(self)-> bool:
        
        return self.__enable_lowercase__
                    

<h1>Test cases:</h1>
<h3> VSM with no lowercase and no punctuation stripping</h3>

In [3]:
vsm = VectorSpaceModel(False , 15, False)

corpus = ['The hotel and the stay were great',
          'This was a great stay',
          'Great stay in a great destination',
          'Great destination']

vsm.fit(corpus)

setup:str = """Stripping enabled: {is_stripping_enabled}
Lowercase enabled: {is_lowercase_enabled}"""

print('\nSetup  Vector space model with:')
print(str.format(setup,is_stripping_enabled=vsm.is_stripping_enabled(), is_lowercase_enabled=vsm.is_lowercase_enabled()))

print('\nVocabulary Created:')
print([word for word in vsm.get_vocabulary().keys()])

      
print('\nVocabulary Statistics:')
print(vsm.get_vocabulary())


Setup  Vector space model with:
Stripping enabled: False
Lowercase enabled: False

Vocabulary Created:
['stay', 'great', 'a', 'Great', 'destination', 'The', 'hotel', 'and', 'the', 'were', 'This', 'was', 'in']

Vocabulary Statistics:
{'stay': Frequency:' 3 idf: 1.2231435513142097 
, 'great': Frequency:' 3 idf: 1.2231435513142097 
, 'a': Frequency:' 2 idf: 1.5108256237659907 
, 'Great': Frequency:' 2 idf: 1.5108256237659907 
, 'destination': Frequency:' 2 idf: 1.5108256237659907 
, 'The': Frequency:' 1 idf: 1.916290731874155 
, 'hotel': Frequency:' 1 idf: 1.916290731874155 
, 'and': Frequency:' 1 idf: 1.916290731874155 
, 'the': Frequency:' 1 idf: 1.916290731874155 
, 'were': Frequency:' 1 idf: 1.916290731874155 
, 'This': Frequency:' 1 idf: 1.916290731874155 
, 'was': Frequency:' 1 idf: 1.916290731874155 
, 'in': Frequency:' 1 idf: 1.916290731874155 
}


<h3> Test case 1 </h3>

In [4]:
first_collection = ['The hotel and the stay were great',
          'This was a great stay',
          'Great stay in a great destination',
          'Great destination']

print(vsm.transform(first_collection))

[[1.2231435513142097, 1.2231435513142097, 0.0, 0.0, 0.0, 1.916290731874155, 1.916290731874155, 1.916290731874155, 1.916290731874155, 1.916290731874155, 0.0, 0.0, 0.0], [1.2231435513142097, 1.2231435513142097, 1.5108256237659907, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.916290731874155, 1.916290731874155, 0.0], [1.2231435513142097, 1.2231435513142097, 1.5108256237659907, 1.5108256237659907, 1.5108256237659907, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.916290731874155], [0.0, 0.0, 0.0, 1.5108256237659907, 1.5108256237659907, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]


<h3> Test case 2 </h3>

In [5]:
second_collection = ['This was a wonderful stay',
               'Dear customer thanks for your review',]

print(vsm.transform(second_collection))

[[1.2231435513142097, 0.0, 1.5108256237659907, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.916290731874155, 1.916290731874155, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]


<h2>VSM with lowercase enabled and no punctuation stripping</h2>

In [6]:
vsm = VectorSpaceModel(True , 15, False)

corpus = ['The hotel and the stay were great',
          'This was a great stay',
          'Great stay in a great destination',
          'Great destination']

vsm.fit(corpus)


setup:str = """Stripping enabled: {is_stripping_enabled}
Lowercase enabled: {is_lowercase_enabled}"""

print('\nSetup Vector space model Details')
print(str.format(setup,is_stripping_enabled=vsm.is_stripping_enabled(), is_lowercase_enabled=vsm.is_lowercase_enabled()))

print('\nVocabulary Created:')
print([word for word in vsm.get_vocabulary().keys()])
      
print('\nVocabulary Statistics:')
print(vsm.get_vocabulary())


Setup Vector space model Details
Stripping enabled: False
Lowercase enabled: True

Vocabulary Created:
['great', 'stay', 'the', 'a', 'destination', 'hotel', 'and', 'were', 'this', 'was', 'in']

Vocabulary Statistics:
{'great': Frequency:' 5 idf: 1.0 
, 'stay': Frequency:' 3 idf: 1.2231435513142097 
, 'the': Frequency:' 2 idf: 1.916290731874155 
, 'a': Frequency:' 2 idf: 1.5108256237659907 
, 'destination': Frequency:' 2 idf: 1.5108256237659907 
, 'hotel': Frequency:' 1 idf: 1.916290731874155 
, 'and': Frequency:' 1 idf: 1.916290731874155 
, 'were': Frequency:' 1 idf: 1.916290731874155 
, 'this': Frequency:' 1 idf: 1.916290731874155 
, 'was': Frequency:' 1 idf: 1.916290731874155 
, 'in': Frequency:' 1 idf: 1.916290731874155 
}


<h3>Test case 3</h3>

In [7]:
first_collection = ['The hotel and the stay were great',
          'This was a great stay',
          'Great stay in a great destination',
          'Great destination']

print(vsm.transform(first_collection))

[[1.0, 1.2231435513142097, 3.83258146374831, 0.0, 0.0, 1.916290731874155, 1.916290731874155, 1.916290731874155, 0.0, 0.0, 0.0], [1.0, 1.2231435513142097, 0.0, 1.5108256237659907, 0.0, 0.0, 0.0, 0.0, 1.916290731874155, 1.916290731874155, 0.0], [2.0, 1.2231435513142097, 0.0, 1.5108256237659907, 1.5108256237659907, 0.0, 0.0, 0.0, 0.0, 0.0, 1.916290731874155], [1.0, 0.0, 0.0, 0.0, 1.5108256237659907, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]


<h3>Test case 4</h3>

In [8]:
second_collection = ['This was a wonderful stay',
               'Dear customer thanks for your review',
              ]

print(vsm.transform(second_collection))

[[0.0, 1.2231435513142097, 0.0, 1.5108256237659907, 0.0, 0.0, 0.0, 0.0, 1.916290731874155, 1.916290731874155, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]


<h2>VSM with lowercase enabled punctuation stripping enabled</h2>

In [9]:
vsm = VectorSpaceModel(True , 15, True)

corpus = ['The hotel and the stay were great',
          'This was a great stay',
          'Great stay in a great destination',
          'Great destination']

vsm.fit(corpus)

setup:str = """Stripping enabled: {is_stripping_enabled}
Lowercase enabled: {is_lowercase_enabled}"""

print('\nSetup Vector space model with:')
print(str.format(setup,is_stripping_enabled=vsm.is_stripping_enabled(), is_lowercase_enabled=vsm.is_lowercase_enabled()))

print('\nVocabulary Created:')
print([word for word in vsm.get_vocabulary().keys()])
      
print('\nVocabulary Statistics:')
print(vsm.get_vocabulary())


Setup Vector space model with:
Stripping enabled: True
Lowercase enabled: True

Vocabulary Created:
['great', 'stay', 'the', 'a', 'destination', 'hotel', 'and', 'were', 'this', 'was', 'in']

Vocabulary Statistics:
{'great': Frequency:' 5 idf: 1.0 
, 'stay': Frequency:' 3 idf: 1.2231435513142097 
, 'the': Frequency:' 2 idf: 1.916290731874155 
, 'a': Frequency:' 2 idf: 1.5108256237659907 
, 'destination': Frequency:' 2 idf: 1.5108256237659907 
, 'hotel': Frequency:' 1 idf: 1.916290731874155 
, 'and': Frequency:' 1 idf: 1.916290731874155 
, 'were': Frequency:' 1 idf: 1.916290731874155 
, 'this': Frequency:' 1 idf: 1.916290731874155 
, 'was': Frequency:' 1 idf: 1.916290731874155 
, 'in': Frequency:' 1 idf: 1.916290731874155 
}


<h3>Test case 5</h3>

In [10]:
first_collection = ['The hotel and the stay were great',
          'This was a great stay',
          'Great stay in a great destination',
          'Great destination']

print(vsm.transform(first_collection))

[[1.0, 1.2231435513142097, 3.83258146374831, 0.0, 0.0, 1.916290731874155, 1.916290731874155, 1.916290731874155, 0.0, 0.0, 0.0], [1.0, 1.2231435513142097, 0.0, 1.5108256237659907, 0.0, 0.0, 0.0, 0.0, 1.916290731874155, 1.916290731874155, 0.0], [2.0, 1.2231435513142097, 0.0, 1.5108256237659907, 1.5108256237659907, 0.0, 0.0, 0.0, 0.0, 0.0, 1.916290731874155], [1.0, 0.0, 0.0, 0.0, 1.5108256237659907, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]


<h3>Test case 6</h3>

In [11]:
second_collection = ['This was a wonderful stay',
               'Dear customer thanks for your review',
              ]

print(vsm.transform(second_collection))

[[0.0, 1.2231435513142097, 0.0, 1.5108256237659907, 0.0, 0.0, 0.0, 0.0, 1.916290731874155, 1.916290731874155, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]


<h1>Assignment 3</h1>
<h2>Part 2</h2>
<h2> VectorSpaceModelExtended class</h2>
<p>
    This class extends <strong>VectorSpaceModel</strong> and overrides <strong>fit()</strong> method

<h4>Following are additional attriutes</h4>
<ul>
    <li>
        <strong>stop_words: </strong> list of stop words to be used
    </li>
    <li>
        <strong>apply_stop_words: </strong> To enable or disable usage of stop words while creating vocabulary
    </li>
</ul>

<h4>Following are new or overridden methods </h4>
<ul>
    <li>
        <strong>fit(): </strong> This method is overridden to add the stop word filter functionality, ultimatley using majority of super methods
    </li>
    <li>
        <strong>is_stop_words_applied(): </strong> Returns if stop words are used in the model
    </li>
    <li>
        <strong>get_stop_words(): </strong>Returns the list of stop words used in the model
    </li>
    <li>
</ul>

<h4>Following dunder methods are also available </h4>
<ul>
    <li>
        <strong>__remove_stop_words__():</strong> Removes stop words from list of words used for vocabulary
    </li>
</ul>
</p>

In [12]:
class VectorSpaceModelExtended(VectorSpaceModel):

    def __init__(self, enable_lowercase:bool, v_max:int, enable_stripping:bool, stop_words:strlist(str), apply_stop_words:bool):
        
        super().__init__(enable_lowercase, v_max, enable_stripping)
        self.__stop_words__:list(str) = stop_words
        self.__apply_stop_words__ = apply_stop_words

    def fit(self, sentences:list[str])-> None:
        
        parsed_words:list[str]= self.__parse_words__(sentences)
        filtered_words:list[str]= self.__remove_stop_words__(parsed_words)
        super().__create_vocabulary_frequency_dictionary__(filtered_words)
        super().__sort_vocabulary__()
        super().__truncate_vocabulary__();
        super().__calculate_vocabulary_idf__(sentences)
        

    def __remove_stop_words__(self, words:list(str))-> list(str):

        if self.__apply_stop_words__:
            return [word for word in words if self.__apply_stop_words__ and word not in self.__stop_words__ ]
        else:
            return words

    def is_stop_words_applied(self)-> bool:
        
        return self.__apply_stop_words__

    def get_stop_words(self)-> list(str):
        return self.__stop_words__
    

<h1>Test cases</h1>
<h2>VSM with lowercase disabled punctuation stripping disabled and stopwords enabled</h2>

In [13]:
stopwords = ["and", "a", "the","it","he","she","where","was","for"]

vsme = VectorSpaceModelExtended(False, 100, False, stopwords, True)

corpus = ['The hotel and the stay were great',
          'This was a great stay',
          'Great stay in a great destination',
          'Great destination']

vsme.fit(corpus)


setup:str = """Stripping enabled: {is_stripping_enabled}
Lowercase enabled: {is_lowercase_enabled}
Stop words applied: {is_stop_words_applied}
Stop words list: {stop_words}"""

print('\nSetup Extended Vector space model with:')
print(str.format(setup,is_stripping_enabled=vsme.is_stripping_enabled(), is_lowercase_enabled=vsme.is_lowercase_enabled(), is_stop_words_applied=vsme.is_stop_words_applied(), stop_words= vsme.get_stop_words()))

print('\nVocabulary Created:')
print([word for word in vsme.get_vocabulary().keys()])

print('\nVocabulary Statistics:')
print(vsme.get_vocabulary())



Setup Extended Vector space model with:
Stripping enabled: False
Lowercase enabled: False
Stop words applied: True
Stop words list: ['and', 'a', 'the', 'it', 'he', 'she', 'where', 'was', 'for']

Vocabulary Created:
['stay', 'great', 'Great', 'destination', 'The', 'hotel', 'were', 'This', 'in']

Vocabulary Statistics:
{'stay': Frequency:' 3 idf: 1.2231435513142097 
, 'great': Frequency:' 3 idf: 1.2231435513142097 
, 'Great': Frequency:' 2 idf: 1.5108256237659907 
, 'destination': Frequency:' 2 idf: 1.5108256237659907 
, 'The': Frequency:' 1 idf: 1.916290731874155 
, 'hotel': Frequency:' 1 idf: 1.916290731874155 
, 'were': Frequency:' 1 idf: 1.916290731874155 
, 'This': Frequency:' 1 idf: 1.916290731874155 
, 'in': Frequency:' 1 idf: 1.916290731874155 
}


<h3>Test case 1</h3>

In [14]:
first_collection = ['The hotel and the stay were great',
          'This was a great stay',
          'Great stay in a great destination',
          'Great destination']

print(vsme.transform(first_collection))

[[1.2231435513142097, 1.2231435513142097, 0.0, 0.0, 1.916290731874155, 1.916290731874155, 1.916290731874155, 0.0, 0.0], [1.2231435513142097, 1.2231435513142097, 0.0, 0.0, 0.0, 0.0, 0.0, 1.916290731874155, 0.0], [1.2231435513142097, 1.2231435513142097, 1.5108256237659907, 1.5108256237659907, 0.0, 0.0, 0.0, 0.0, 1.916290731874155], [0.0, 0.0, 1.5108256237659907, 1.5108256237659907, 0.0, 0.0, 0.0, 0.0, 0.0]]


<h3>Test case 2</h3>

In [15]:
second_collection = ['This was a wonderful stay',
               'Dear customer thanks for your review',
              ]

print(vsme.transform(second_collection))

[[1.2231435513142097, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.916290731874155, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]


In [16]:
vsme.transform(second_collection)

[[1.2231435513142097, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.916290731874155, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]